# Python: не только `print` и циклы

Этот ноутбук — мини-выставка. Один и тот же язык умеет рисовать математику, крутить симуляции, разбирать таблицы и собирать картинки «из ничего».

Запускай ячейки сверху вниз. Все графики собраны из **чисел и нескольких строк кода** — без Photoshop.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.facecolor": "#0b1020",
    "axes.facecolor": "#0b1020",
    "axes.edgecolor": "#8aa0c8",
    "axes.labelcolor": "#dce6ff",
    "xtick.color": "#9bb0d4",
    "ytick.color": "#9bb0d4",
    "text.color": "#e8eefc",
    "axes.titlecolor": "#ffffff",
    "savefig.facecolor": "#0b1020",
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. Математика, которая выглядит как искусство

Параметрические кривые: задаём `x(t)` и `y(t)`, а Python рисует траекторию. Меняй коэффициенты — получится совсем другая фигура.

In [ ]:
t = np.linspace(0, 2 * np.pi, 4000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))

pairs = [
    (3, 2, "Лиссажу 3:2"),
    (5, 4, "Лиссажу 5:4"),
    (7, 3, "Лиссажу 7:3"),
]
for ax, (a, b, title) in zip(axes, pairs):
    x = np.sin(a * t)
    y = np.sin(b * t + np.pi / 4)
    ax.scatter(x, y, c=t, cmap="magma", s=0.4, linewidths=0)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")

fig.suptitle("Кривые Лиссажу — осциллограф в Jupyter", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
theta = np.linspace(0, 24 * np.pi, 8000)
r = np.exp(0.08 * theta)  # логарифмическая спираль
x = r * np.cos(theta)
y = r * np.sin(theta)

k = 7  # лепестки розы
rose_r = np.cos(k * theta)
rx, ry = rose_r * np.cos(theta), rose_r * np.sin(theta)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
ax1.scatter(x, y, c=theta, cmap="turbo", s=0.3, linewidths=0)
ax1.set_title("Логарифмическая спираль (как у раковины)")
ax1.set_aspect("equal")
ax1.axis("off")

ax2.scatter(rx, ry, c=np.abs(rose_r), cmap="plasma", s=0.25, linewidths=0)
ax2.set_title(f"Роза Гвидо: r = cos({k}θ)")
ax2.set_aspect("equal")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 2. Фрактал: множество Мандельброта

Для каждой точки плоскости `c` крутим формулу $z \mapsto z^2 + c$. Если точка «не убегает» — она внутри множества. Картинка бесконечно детальная: чем ближе зум, тем больше завитков.

In [ ]:
def mandelbrot(xmin, xmax, ymin, ymax, width=900, height=700, max_iter=80):
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    c = x[None, :] + 1j * y[:, None]
    z = np.zeros_like(c)
    escape = np.zeros(c.shape, dtype=int)
    for i in range(max_iter):
        mask = np.abs(z) <= 2
        z[mask] = z[mask] ** 2 + c[mask]
        escape[mask] = i
    return escape

img = mandelbrot(-2.2, 0.8, -1.2, 1.2)

fig, ax = plt.subplots(figsize=(10, 7.4))
ax.imshow(img, cmap="twilight_shifted", origin="lower",
          extent=[-2.2, 0.8, -1.2, 1.2])
ax.set_title("Множество Мандельброта")
ax.set_xlabel("Re(c)")
ax.set_ylabel("Im(c)")
plt.tight_layout()
plt.show()

## 3. Хаос в 3D: аттрактор Лоренца

Три простых дифференциальных уравнения — и траектория никогда не повторяется. Два «крыла бабочки»: отсюда метафора про бабочку, взмах которой меняет погоду.

In [ ]:
def lorenz(n=12000, dt=0.008, sigma=10.0, rho=28.0, beta=8 / 3):
    xs = np.empty(n)
    ys = np.empty(n)
    zs = np.empty(n)
    x = y = z = 1.0
    for i in range(n):
        xs[i], ys[i], zs[i] = x, y, z
        x, y, z = (
            x + dt * sigma * (y - x),
            y + dt * (x * (rho - z) - y),
            z + dt * (x * y - beta * z),
        )
    return xs, ys, zs

xs, ys, zs = lorenz()

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor("#0b1020")
fig.patch.set_facecolor("#0b1020")
sc = ax.scatter(xs, ys, zs, c=np.arange(len(xs)), cmap="cool", s=0.4, linewidths=0)
ax.set_title("Аттрактор Лоренца")
ax.set_axis_off()
ax.view_init(elev=22, azim=120)
plt.tight_layout()
plt.show()

## 4. Симуляция: случайное блуждание тысячи частиц

Каждая точка на каждом шаге делает крошечный шаг в случайную сторону. Вместе они рисуют «дымку» — так моделируют диффузию, пыльцу в воде и даже цены на рынке (очень грубо).

In [ ]:
rng = np.random.default_rng(7)
n_particles, n_steps = 900, 180
steps = rng.normal(0, 0.08, size=(n_steps, n_particles, 2))
paths = np.cumsum(steps, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, t, title in zip(axes, [20, 80, 179], ["через 20 шагов", "через 80", "через 180"]):
    pts = paths[t]
    r = np.hypot(pts[:, 0], pts[:, 1])
    ax.scatter(pts[:, 0], pts[:, 1], c=r, cmap="inferno", s=8, alpha=0.85, linewidths=0)
    ax.set_title(title)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle("Диффузия: облако расползается", y=1.03)
plt.tight_layout()
plt.show()

## 5. Игра «Жизнь» Конвея

Клетка живёт или умирает по трём правилам. Никакой случайности после старта — и всё равно появляются глайдеры, осцилляторы и хаос. Python считает поколение за поколение за миллисекунды.

In [ ]:
rng = np.random.default_rng(42)
grid = rng.random((90, 140)) > 0.72

def life_step(g):
    n = sum(np.roll(np.roll(g, i, 0), j, 1)
            for i in (-1, 0, 1) for j in (-1, 0, 1))
    n -= g
    return (n == 3) | (g & (n == 2))

snapshots = []
g = grid.copy()
for step in range(80):
    if step in (0, 12, 40, 79):
        snapshots.append((step, g.copy()))
    g = life_step(g)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (step, g) in zip(axes, snapshots):
    ax.imshow(g, cmap="cubehelix", interpolation="nearest")
    ax.set_title(f"поколение {step}")
    ax.axis("off")
fig.suptitle("Клеточный автомат: жизнь из правил", y=1.05)
plt.tight_layout()
plt.show()

## 6. Картинка из формул: поле и шум

Слева — линии тока векторного поля (как ветер). Справа — процедурный «мрамор»: никаких фото, только синусы и шум.

In [ ]:
Y, X = np.mgrid[-3:3:28j, -3:3:28j]
U = -Y - 0.3 * X
V = X - 0.3 * Y
speed = np.hypot(U, V)

xs = np.linspace(-2.2, 2.2, 700)
ys = np.linspace(-1.4, 1.4, 450)
XX, YY = np.meshgrid(xs, ys)
marble = np.sin(5 * XX + 3 * np.sin(4 * YY) + 1.5 * np.sin(7 * XX * YY))
marble += 0.35 * np.sin(18 * XX - 11 * YY)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.streamplot(X, Y, U, V, color=speed, cmap="plasma", density=1.35, linewidth=1.1)
ax1.set_title("Векторное поле (поток)")
ax1.set_aspect("equal")
ax1.axis("off")

ax2.imshow(marble, cmap="RdYlBu_r", origin="lower", extent=[xs.min(), xs.max(), ys.min(), ys.max()])
ax2.set_title("Процедурный мрамор")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 7. Звук без колонок: спектр аккорда

Нота — это синус. Аккорд — сумма синусов. Преобразование Фурье раскладывает смесь обратно на частоты: пики на графике — это «ноты», которые Python услышал в числе.

In [ ]:
sr = 8000
t = np.linspace(0, 1.0, sr, endpoint=False)
# C-major: C4, E4, G4
chord = (np.sin(2 * np.pi * 261.63 * t)
         + 0.7 * np.sin(2 * np.pi * 329.63 * t)
         + 0.5 * np.sin(2 * np.pi * 392.00 * t))

freq = np.fft.rfftfreq(len(chord), 1 / sr)
spec = np.abs(np.fft.rfft(chord))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), gridspec_kw={"height_ratios": [1, 1.2]})
axes[0].plot(t[:800], chord[:800], color="#7ee0ff", lw=1.4)
axes[0].set_title("Волна аккорда C–E–G (40 мс)")
axes[0].set_xlabel("время, с")

axes[1].plot(freq, spec, color="#ffb347", lw=1.5)
axes[1].set_xlim(0, 800)
axes[1].set_title("Спектр: три пика — три ноты")
axes[1].set_xlabel("частота, Гц")
for f, name in [(261.63, "C4"), (329.63, "E4"), (392.00, "G4")]:
    axes[1].axvline(f, color="#ff6b9d", alpha=0.5, ls="--")
    axes[1].text(f + 8, spec.max() * 0.85, name, color="#ff6b9d")
plt.tight_layout()
plt.show()

## 8. Живые данные: школьники из `data.csv`

Тысяча строк — возраст, балл по математике, кружок. Pandas читает таблицу за одну строку, matplotlib показывает, что в ней спрятано.

In [ ]:
df = pd.read_csv("data.csv")
display(df.head())
print(f"строк: {len(df)} | кружки: {', '.join(sorted(df['club'].unique()))}")

In [ ]:
palette = {
    "спорт": "#ff6b6b",
    "информатика": "#4ecdc4",
    "музыка": "#ffe66d",
    "нет": "#9aa4c7",
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))

for club, color in palette.items():
    part = df[df["club"] == club]
    axes[0].scatter(part["age"], part["math"], c=color, s=18, alpha=0.55, label=club, edgecolors="none")
axes[0].set_xlabel("возраст")
axes[0].set_ylabel("балл по математике")
axes[0].set_title("Каждый кружок — свой цвет")
axes[0].legend(frameon=False, fontsize=9)

means = df.groupby("club")["math"].mean().reindex(palette)
axes[1].bar(means.index, means.values, color=[palette[c] for c in means.index])
axes[1].set_title("Средний балл по кружкам")
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=20)

axes[2].hist(df["math"], bins=20, color="#7ee0ff", edgecolor="#0b1020")
axes[2].set_title("Распределение баллов")
axes[2].set_xlabel("math")

plt.tight_layout()
plt.show()

## 9. Анимация: орбиты

Не снимок, а движение. Ниже — несколько точек на эллипсах (как планеты). В Jupyter крутится прямо в ячейке.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.2))
ax.set_xlim(-2.3, 2.3)
ax.set_ylim(-2.3, 2.3)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Мини-солнечная система")

thetas = np.linspace(0, 2 * np.pi, 400)
orbits = [(1.0, 0.85, "#ffd166"), (1.45, 1.15, "#06d6a0"), (1.95, 1.55, "#118ab2")]
for a, b, c in orbits:
    ax.plot(a * np.cos(thetas), b * np.sin(thetas), color=c, alpha=0.35, lw=1.2)

sun = ax.scatter([0], [0], s=220, c="#ff9f1c", zorder=5)
planets = [ax.scatter([], [], s=s, c=c, zorder=6) for s, c in [(45, "#ffd166"), (70, "#06d6a0"), (55, "#118ab2")]]
trails = [ax.plot([], [], color=c, lw=1.0, alpha=0.7)[0] for _, _, c in orbits]
hist = [[] for _ in orbits]

def frame(k):
    t = k * 0.08
    speeds = [1.6, 1.05, 0.7]
    artists = []
    for i, ((a, b, _), p, trail, h, w) in enumerate(zip(orbits, planets, trails, hist, speeds)):
        x, y = a * np.cos(w * t), b * np.sin(w * t)
        p.set_offsets([[x, y]])
        h.append((x, y))
        h[:] = h[-40:]
        trail.set_data([q[0] for q in h], [q[1] for q in h])
        artists += [p, trail]
    return artists

anim = animation.FuncAnimation(fig, frame, frames=180, interval=35, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## Что из этого следует

Python — это **клей**: одни и те же идеи (массивы, циклы, функции) работают в очень разных мирах.

| Хочешь… | Инструмент |
|---|---|
| быстро считать массивы | `numpy` |
| таблицы как Excel, но программно | `pandas` |
| графики и анимации | `matplotlib` |
| нейросети, игры, сайты, роботы | другие библиотеки на том же языке |

На курсе мы начнём с простых программ, но потолок языка — примерно вот такой: от формулы на салфетке до картинки, симуляции и разбора настоящих данных.